# Lab 2: the training pipeline

This lab builds the pipeline that trains a forecast model: a Dataset that serves standardised tensors, a loss, a Vision Transformer, the LightningModule that holds them and defines the training and validation steps, and the configuration that names every choice.
Each piece goes into its module under `utils/`, in the order below, and section 6 lists the tests each piece has to pass before the training run at the end.
Steps 1 to 4 take their configuration as the dataclasses of step 5, so write `utils/config.py` alongside, field by field, as each step names its fields.

Conventions: tensors carry the axes `(batch, variable, time, latitude, longitude)` in that order, every reshape is a named einops pattern, and every hyperparameter lives in a config field.

In [1]:
from pathlib import Path

import xarray as xr
import torch
import lightning as L

xr.set_options(keep_attrs=True, display_expand_data=False, use_bottleneck=False)
torch.manual_seed(0)

### Documentation

- [torch.utils.data.Dataset](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.Dataset): `__len__` and `__getitem__`.
- [torch.nn.Module](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html): `__init__` registers the parameters, `forward` computes.
- [einops.einsum](https://einops.rocks/api/einsum/) and [EinMix](https://einops.rocks/3-einmix-layer/): named-axis contractions and named-axis linear layers.
- [LightningModule](https://lightning.ai/docs/pytorch/stable/common/lightning_module.html): `training_step`, `validation_step`, `configure_optimizers`, and the dataloader hooks.
- [dataclasses](https://docs.python.org/3/library/dataclasses.html) and [PyYAML](https://pyyaml.org/wiki/PyYAMLDocumentation): the configuration and its file form.

### Data on disk

The pipeline reads from two local zarr stores, the fields and their statistics, so the cloud store is read once.
The cell below writes both into `data/`: the nine variables of lab 1 at 5.625 degrees over `PERIOD`, and the global mean and standard deviation per variable with a `statistic` dimension.
Shorten `PERIOD` on a slow connection; a year of a level variable is about 155 MB of download.

In [3]:
DATA = Path("data")
DATA.mkdir(exist_ok=True)
ZARR = DATA / "era5_5p6.zarr"
STATS = DATA / "stats.zarr"

STORE = "gs://weatherbench2/datasets/era5/1959-2023_01_10-6h-64x32_equiangular_conservative.zarr"
VARIABLES = {
    "T2M":  ("2m_temperature", None),
    "U10M": ("10m_u_component_of_wind", None),
    "V10M": ("10m_v_component_of_wind", None),
    "TP6h": ("total_precipitation_6hr", None),
    "Z500": ("geopotential", 500),
    "T850": ("temperature", 850),
    "Q700": ("specific_humidity", 700),
    "U250": ("u_component_of_wind", 250),
    "V250": ("v_component_of_wind", 250),
}
PERIOD = slice("2015", "2019")

if not ZARR.exists():
    era5 = xr.open_zarr(STORE, storage_options={"token": "anon"}, chunks={}).sel(time=PERIOD)
    fields = {k: era5[name].sel(level=level, drop=True) if level is not None else era5[name] for k, (name, level) in VARIABLES.items()}
    data = xr.Dataset(fields).transpose("time", "latitude", "longitude").chunk({"time": 100})
    data.to_zarr(ZARR, mode="w", zarr_format=2)
if not STATS.exists():
    data = xr.open_zarr(ZARR)
    stats = xr.concat([data.mean(), data.std()], dim=xr.DataArray(["mean", "std"], dims="statistic", name="statistic")).compute()
    stats.to_zarr(STATS, mode="w")

print(xr.open_zarr(ZARR))
print(xr.open_zarr(STATS).to_dataframe().T)

<xarray.Dataset> Size: 539MB
Dimensions:    (time: 7304, latitude: 32, longitude: 64)
Coordinates:
  * time       (time) datetime64[ns] 58kB 2015-01-01 ... 2019-12-31T18:00:00
  * latitude   (latitude) float64 256B -87.19 -81.56 -75.94 ... 81.56 87.19
  * longitude  (longitude) float64 512B 0.0 5.625 11.25 ... 343.1 348.8 354.4
Data variables:
    Q700       (time, latitude, longitude) float32 60MB dask.array<chunksize=(100, 32, 64), meta=np.ndarray>
    T2M        (time, latitude, longitude) float32 60MB dask.array<chunksize=(100, 32, 32), meta=np.ndarray>
    T850       (time, latitude, longitude) float32 60MB dask.array<chunksize=(100, 32, 64), meta=np.ndarray>
    TP6h       (time, latitude, longitude) float32 60MB dask.array<chunksize=(100, 32, 32), meta=np.ndarray>
    U10M       (time, latitude, longitude) float32 60MB dask.array<chunksize=(100, 32, 32), meta=np.ndarray>
    U250       (time, latitude, longitude) float32 60MB dask.array<chunksize=(100, 32, 64), meta=np.ndarray>


/data/kovganvl/miniconda3/envs/dlwp/lib/python3.13/site-packages/zarr/api/asynchronous.py:246: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [20]:
data = xr.open_zarr(ZARR)
data['Q700'].values.shape

(7304, 32, 64)

### einops in five minutes

Three functions and one layer, on a few fields from the store.
The examples follow the einops tutorial by Alex Rogozhnikov, [part 1, basics](https://einops.rocks/1-einops-basics/) and [part 3, EinMix](https://einops.rocks/3-einmix-layer/), with weather fields in place of its images.
A pattern names every axis on the left and on the right; an axis that appears on both sides is kept, an axis in parentheses is composed of or decomposed into the named parts, and the sizes of the parts are given as keyword arguments where they cannot be inferred.

In [4]:
from einops import rearrange, reduce, repeat, einsum
from einops.layers.torch import EinMix

x = torch.from_numpy(xr.open_zarr(ZARR)["T2M"].isel(time=slice(0, 4)).values)   # (t, h, w): four states of T2M
x.shape

torch.Size([4, 32, 64])

In [5]:
# rearrange: transposition, composition, and decomposition of axes; the number of elements never changes
print(rearrange(x, "t h w -> t w h").shape)                 # transpose
print(rearrange(x, "t h w -> (t h) w").shape)               # compose: the four fields stacked along the height
print(rearrange(x, "(a b) h w -> a b h w", a=2).shape)      # decompose: 4 = 2 * 2, a and b are new axes
print(rearrange(x, "t h w -> t 1 h w").shape)               # a new axis of length one

torch.Size([4, 64, 32])
torch.Size([128, 64])
torch.Size([2, 2, 32, 64])
torch.Size([4, 1, 32, 64])


In [6]:
# the ViT's own reshape: a field becomes a sequence of patches, each patch a vector of its values
patches = rearrange(x, "t (h hh) (w ww) -> t (h w) (hh ww)", hh=4, ww=4)
print(patches.shape)                                        # (t, 8 * 16 patches, 4 * 4 values)
back = rearrange(patches, "t (h w) (hh ww) -> t (h hh) (w ww)", h=8, hh=4, ww=4)
print(torch.equal(back, x))                                 # and back, exactly

torch.Size([4, 128, 16])
True


In [7]:
# reduce: the same syntax with a reduction; an axis missing on the right is reduced over
print(reduce(x, "t h w -> h w", "mean").shape)                              # the time mean
print(reduce(x, "t (h hh) (w ww) -> t h w", "mean", hh=4, ww=4).shape)     # mean pooling by 4 by 4 patches
print(reduce(x, "t h w -> t () ()", "max").shape)                           # one maximum per field, axes kept for broadcasting

torch.Size([32, 64])
torch.Size([4, 8, 16])
torch.Size([4, 1, 1])


In [8]:
# repeat: the opposite of reduce; an axis new on the right is repeated
mean_field = reduce(x, "t h w -> h w", "mean")
print(repeat(mean_field, "h w -> t h w", t=4).shape)                        # broadcast the mean over time
weights = torch.cos(torch.deg2rad(torch.linspace(-87.1875, 87.1875, 32)))
print(repeat(weights, "h -> h w", w=64).shape)                              # a latitude weight for every grid point

torch.Size([4, 32, 64])
torch.Size([32, 64])


In [9]:
# einsum: a contraction over named axes; the pattern names what is summed over
similarity = einsum(patches, patches, "t i d, t j d -> t i j")            # every patch against every patch, per field
print(similarity.shape)                                                     # this is the shape of an attention matrix

torch.Size([4, 128, 128])


In [10]:
# EinMix: a linear layer between named axes; the weight has the axes you name, and the mixing follows the pattern
# the patch embedding of a Vision Transformer, as in the einops tutorial: every patch of hh by ww values becomes a vector of length d
to_tokens = EinMix("t (h hh) (w ww) -> t (h w) d", weight_shape="hh ww d", hh=4, ww=4, d=16)
tokens = to_tokens(x)
print(tokens.shape, "with", sum(p.numel() for p in to_tokens.parameters()), "parameters")   # 4 * 4 * 16 weights
# torch.nn.Linear on the last axis is the special case  EinMix("... i -> ... o", weight_shape="i o")

torch.Size([4, 128, 16]) with 256 parameters


## 1. Dataset (`utils/dataset.py`)

A `torch.utils.data.Dataset` holds the standardised fields as one tensor in memory and returns a window of consecutive time steps per index; the DataLoader batches those windows.

`WeatherDataset(config: DatasetConfig)`, with the config fields `path`, `stats_path`, `variables`, `sequence_length`, `time_slice`, `lat_slice`, `lon_slice`, and, for the DataLoader of step 4, `batch_size` and `num_workers`:

- `__init__` opens the zarr at `path`, selects `time_slice`, `lat_slice`, and `lon_slice` (each a dict with `start`, `stop`, `step`, or `None` for everything), and selects `variables` in the config's order.
  It asserts that `ds.to_array(dim="variable")` carries the variables in exactly that order.
  It reads the mean and standard deviation of every variable from the zarr store at `stats_path`, keeps the sliced xarray Dataset, the means, and the standard deviations as attributes, and holds `to_tensor(ds)` of the sliced Dataset as its data.
- `to_tensor(ds)` and `to_xarray(x, **coords)` are the two directions of one conversion.
  `to_tensor` takes an xarray Dataset, selects the config's variables in order, standardises each with its mean and standard deviation, and stacks them on a leading `variable` axis into one float32 tensor `(variable, time, latitude, longitude)` (NaN to 0).
  `to_xarray` takes a standardised tensor `(variable, latitude, longitude)`, or `(variable, n, latitude, longitude)` with the middle axis named and given its coordinate values by the one keyword argument (`time=...` or `prediction_timedelta=...`), undoes the standardisation per variable, and returns a Dataset with the variables as data variables and `latitude` and `longitude` from the dataset.
- `__len__` is the number of windows of `sequence_length` consecutive steps: `time - sequence_length + 1`.
- `__getitem__(idx)` returns `tensor[:, idx : idx + sequence_length]`, shape `(variable, sequence_length, latitude, longitude)`.

## 2. Loss (`utils/loss_fn.py`)

The loss is a `torch.nn.Module`, built and configured like every other piece; here it is the mean squared error, with a signature that already accepts the keyword arguments of the later losses.

`MSE(torch.nn.Module)`: `forward(prediction, target, **kwargs)` returns the mean of `(prediction - target) ** 2` over every axis, as one scalar; a `weight` in `kwargs`, broadcastable to the prediction, multiplies the squared error before the mean, and other keyword arguments are ignored.

## 3. Architecture (`utils/components.py`)

A Vision Transformer cuts the field into patches, embeds every patch as one token, mixes the tokens through transformer blocks, and reads every token back out as a patch.
All linear maps between named axes are `EinMix` layers, and the attention itself is written out.

Config fields (`NetworkConfig`): `name` (`"vit"` or `"persistence"`), `dim`, `num_layers`, `num_heads`, `dim_heads`, `patch_size` as `(hh, ww)`, `expansion_factor`.

- `FFN(dim, expansion_factor)`: the gated feed-forward unit: one linear map from `dim` to twice the hidden width `dim * expansion_factor`, split into two halves, the SiLU of one half times the other, and one linear map back to `dim`.
- `MHSA(dim, num_heads, dim_heads)`: multi-head self-attention, written out.
  Three EinMix projections take the tokens `(batch, tokens, dim)` to queries, keys, and values of shape `(batch, heads, tokens, dim_heads)`; the attention matrix, `(batch, heads, tokens, tokens)`, is the softmax over the keys of the query-key product scaled by `1 / sqrt(dim_heads)`, as an einsum over named axes; the output is the attention matrix applied to the values, and one EinMix projection back to `(batch, tokens, dim)`.
- `TransformerBlock(dim, num_heads, dim_heads, expansion_factor)`: pre-norm residual block, attention then feed-forward, with a `torch.nn.RMSNorm` in front of each.
- `ViT(config, num_variables, field_size)`: `to_tokens`, an EinMix from fields `(batch, variable, H, W)` to tokens `(batch, h * w, dim)` with one patch of `patch_size` per token; a learned positional embedding `(h * w, dim)` added to the tokens; `num_layers` blocks; a final norm; and `to_fields`, the EinMix from tokens back to fields.
  `forward(x)` maps `(b, v, H, W)` to `(b, v, H, W)`.
- `Persistence()`: `forward(x)` returns `x`.

## 4. LightningModule (`utils/lightning_module.py`)

The LightningModule holds the datasets, the network, and the loss, and defines the training step, the validation step, and the optimiser; `lightning.Trainer` runs the loops.

`ForecastModule(L.LightningModule)`, built from the full `Config` of step 5:

- `__init__` builds the training dataset from `config.dataset`, the validation dataset from the same config with `time_slice = config.trainer.val_time_slice` and `sequence_length = config.trainer.rollout_steps + 1` (`dataclasses.replace`), the network named by `config.network.name` (`ViT` with `num_variables` and `field_size` read off the dataset, or `Persistence`), and the loss (`MSE` with `config.objective.kwargs`).

- `training_step(batch, batch_idx)`: `x = batch[:, :, 0]`, `y = batch[:, :, 1]`, `loss = self.loss(self.model(x), y)`, logged as `train/loss` and returned.
  The training step is single-step; the roll-out is in the validation step only.

- `validation_step(batch, batch_idx)`: starts from `batch[:, :, 0]`, applies the model `rollout_steps` times, feeding each prediction back in, and logs the loss of step `k` (`k = 1, ..., rollout_steps`) against `batch[:, :, k]` as `val/loss_step{k}`.

- `forecast(x, steps)`: the same roll-out as a method, returning the predictions stacked on a new time axis, `(b, v, steps, H, W)`.

- `configure_optimizers`: `AdamW` with `lr`, `weight_decay`, and `betas` from `config.trainer`, and a `CosineAnnealingLR` over `max_steps` stepped per batch (`{"optimizer": ..., "lr_scheduler": {"scheduler": ..., "interval": "step"}}`).

- `train_dataloader` and `val_dataloader`: `DataLoader` with `batch_size` and `num_workers` from `config.dataset`; the training loader shuffles.

Config fields (`TrainerConfig`): `lr`, `weight_decay`, `betas`, `max_steps`, `rollout_steps`, `val_time_slice`.

## 5. Configuration (`utils/config.py`)

One dataclass per piece gathers the keyword arguments named above, and one `Config` gathers the four; a yaml file is the same structure on disk.

- `DatasetConfig`, `NetworkConfig`, `ObjectiveConfig` (`name` and `kwargs`; `name` is `"mse"` for now and carried for the later losses), and `TrainerConfig`, with the fields listed in steps 1 to 4 and defaults where a default is sensible.
- `Config` with the fields `dataset`, `network`, `objective`, `trainer`; `Config.from_dict(cfg)` builds each dataclass from the matching sub-dictionary, `Config.from_yaml(path)` reads the file with `yaml.safe_load` and calls `from_dict`, and `to_dict` (`dataclasses.asdict`) is what the LightningModule saves as its hyperparameters.

`configs/vit.yaml` in the repository is the configuration of the run at the end, with every field above, and `configs/persistence.yaml` is the same pipeline with persistence as the network; paths in them are relative to the repository root.

In [21]:
from utils.config import Config

config = Config.from_yaml("configs/vit.yaml")
config

Config(dataset=DatasetConfig(path='data/era5_5p6.zarr', stats_path='data/stats.zarr', variables=['T2M', 'U10M', 'V10M', 'TP6h', 'Z500', 'T850', 'Q700', 'U250', 'V250'], sequence_length=2, time_slice={'start': '2015', 'stop': '2018'}, lat_slice=None, lon_slice=None, batch_size=16, num_workers=0), network=NetworkConfig(name='vit', dim=128, num_layers=4, num_heads=4, dim_heads=32, patch_size=[4, 4], expansion_factor=2), objective=ObjectiveConfig(name='mse', kwargs={}), trainer=TrainerConfig(lr=0.001, weight_decay=0.01, betas=[0.9, 0.95], max_steps=500, rollout_steps=4, val_time_slice={'start': '2019', 'stop': '2019'}))

## 6. Tests

The pieces are tested as a pipeline, from configuration files, and the checks are on properties rather than on one setting: shapes that follow the configuration, runs that complete from every configuration, and numbers of the expected size.
Each check is a few lines in a cell: build, run, assert.

1. Shapes follow the configuration.
   For any `DatasetConfig`, a sample has shape `(len(variables), sequence_length, latitude, longitude)` of the selected region, and the length is the number of time steps minus `sequence_length` plus one.
   The ViT maps a state to a state of the same shape for any patch size that divides the field.
   Over the training period a standardised field has mean near 0 and standard deviation near 1, and `to_xarray(to_tensor(ds))` returns `ds` to float32 precision.
2. The pipeline runs from every configuration file unchanged.
   `configs/vit.yaml` and `configs/persistence.yaml` are given; write a third of your own (a regional crop, fewer variables, another patch size or `sequence_length`, another `rollout_steps`) and run `Trainer.validate` and a few training steps from each.
   Between runs only the file changes.
3. Numbers are of the expected size.
   In standardised units the loss of a freshly initialised ViT is of order 1; the persistence loss at six hours is of order 0.1 and grows with the roll-out step; a ViT after a few hundred steps sits below persistence at step 1, and its losses grow with the step as well.
   A loss of exactly 0, a negative loss, a NaN, or a loss far above 1 points at the standardisation, the target index, or the learning rate.
4. The way back is exact for persistence.
   Rolled out from a validation state and converted with `to_xarray(..., prediction_timedelta=leads)`, the first lead equals the field stored in the zarr at that time to float32 precision and in physical units (`T2M` in the 200s of kelvin), for every variable.

With the tests passing, the same `Trainer` trains the ViT.
A few hundred steps run in minutes on a laptop; the validation losses per roll-out step are read against the persistence values of test 3.

In [23]:
from utils.lightning_module import ForecastModule

module = ForecastModule(config)
trainer = L.Trainer(max_steps=config.trainer.max_steps, accelerator="auto", logger=False, enable_checkpointing=False,
                    val_check_interval=100, limit_val_batches=20)
trainer.fit(module)
print({k: round(float(v), 4) for k, v in trainer.callback_metrics.items()})

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type | Params | Mode  | FLOPs
-----------------------------------------------
0 | model | ViT  | 709 K  | train | 0    
1 | loss  | MSE  | 0      | train | 0    
-----------------------------------------------
709 K     Trainable params
0         Non-trainable params
709 K     Total params
2.839     Total estimated model params size (MB)
46        Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/data/kovganvl/miniconda3/envs/dlwp/lib/python3.13/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/data/kovganvl/miniconda3/envs/dlwp/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.
/data/kovganvl/miniconda3/envs/dlwp/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=500` reached.


{'train/loss': 0.0857, 'val/loss_step1': 0.0863, 'val/loss_step2': 0.1565, 'val/loss_step3': 0.2252, 'val/loss_step4': 0.2879}


## 7. Back to the verification of lab 1

The validation losses above are standardised mean squared errors, one number over all variables.
The verification of lab 1, section 5, scores a forecast per variable, in physical units, area-weighted, and against the climatology and persistence; the trained model belongs on those curves.

- From the validation dataset, take the states at 00 UTC of 2019 as initialisations, roll each out with `forecast(x, steps)` for a lead range beyond the training length (eight steps, two days, is enough to see the trend), and do the same with a `Persistence` module.
- Convert every roll-out with `to_xarray(..., prediction_timedelta=leads)` and concatenate along the initialisation time into the forecast schema of lab 1, `(time, prediction_timedelta, latitude, longitude)`; the initialisation times are those of the validation dataset's xarray Dataset.
- Align the truth from the zarr at `time + prediction_timedelta`, as in lab 1, and score both forecasts with the functions of lab 1, section 5: RMSE and ACC against lead time for `Z500` and `T2M`, on one figure with the climatological level, and the skill score of the ViT against persistence.
- Where the ViT beats persistence and where it does not, at which lead the two cross, and how the two-day roll-out compares with the four validation steps are what to read off.